# Week 5

### Goals
- Expose controlled Python functions to the model.
- Validate tool arguments and results.

### Tasks
- Implement search_movies(), filter_movies(), get_movie_details() and rank_candidates().
- Define clear tool names, parameter descriptions and return schemas.
- Connect the tools to the local model’s supported tool-calling interface, or use validated structured action
selection if necessary.
- Reject invalid parameters, nonexistent movie IDs and unsupported claims.
- Test each tool directly before testing the full agent.

### Expected Output
- 05_tools.ipynb and reusable functions in src/.
- Examples showing successful and rejected tool calls.

---
The first two task are completet in the file `src/tools.py`. The relevant functions are written into the .py file so they can be imported and reused as tools by the ollama agent. The extensive testing of the written functions on the cleaned movie data set (from week 3) follows in the code cells below.

In [8]:
# Import relevant libraries and modules from src/tools.py
import pandas as pd
from ollama import chat

import sys
sys.path.append("../")
from src.tools import search_movies, filter_movies, rank_candidates, get_movie_details

### Test python tool logic

In [9]:
# Test filter and rank
print("\n--- Filtering & Ranking ---")
action_movies = filter_movies(genre='Action', year=1999)
ranked_action = rank_candidates(action_movies, top_n=2)
print(f"Top 1999 Action Movies: {[m['title'] for m in ranked_action]}")

# Test search
print("\n--- Searching ---")
matrix_search = search_movies(query='Matrix')
print(f"Search Results for 'Matrix': {[m['title'] for m in matrix_search]}")

# Test details and invalid ID rejection
print("\n--- Details & Rejection ---")
valid_details = get_movie_details(movie_id=2571) # The Matrix
print(f"Valid Details (Notice Unknowns): {valid_details.get('title')} | Runtime: {valid_details.get('runtime')}")

invalid_details = get_movie_details(movie_id=9999999)
print(f"Invalid ID Handling: {invalid_details}")


--- Filtering & Ranking ---
Top 1999 Action Movies: ['Fight Club', 'The Matrix']

--- Searching ---
Search Results for 'Matrix': ['The Matrix', 'The Matrix Reloaded', 'The Matrix Revolutions', 'The Animatrix']

--- Details & Rejection ---
Valid Details (Notice Unknowns): The Matrix | Runtime: Unknown
Invalid ID Handling: {'error': 'No movie found with ID 9999999.'}


### Test LLM tool connection

#### Simple test to check tool and argument selection

In [10]:
ALL_TOOLS = [search_movies, filter_movies, rank_candidates, get_movie_details]

messages = [{'role': 'user', 'content': 'Can you find me some top rated Sci-Fi movies from 1999?'}]

response = chat(
    model='llama3.2',
    messages=messages,
    tools=ALL_TOOLS
)

print("\nLLM Decision:")
if response.message.tool_calls:
    for tc in response.message.tool_calls:
        print(f"Tool Selected: {tc.function.name}")
        print(f"Arguments Provided: {tc.function.arguments}")
else:
    print("The model failed to select a tool.")


LLM Decision:
Tool Selected: filter_movies
Arguments Provided: {'year': 1999, 'genre': 'Sci-Fi'}


**Observations:**   
Simple tool selection tets seems to be successful. The agent correctly called the `filter_movie()` function with the arguments 'genre': 'Sci-Fi', 'year': '1999' isolated from the user request. This test might be a bit too small since it would be logical for the llm to call a second tool `rank_candidate()` to specifically search for the top rated movies since the request also asked for that.

#### More extensive LLM test

In [5]:
def run_real_agent_test(user_prompt):
    messages = [{'role': 'user', 'content': user_prompt}]
    # Map the string names Ollama returns to the actual Python functions
    tool_map = {f.__name__: f for f in ALL_TOOLS}
    step_count = 0
    
    print(f"\n=== TRACE FOR: '{user_prompt}' ===")
    
    while step_count < 4:
        step_count += 1
        response = chat(model='llama3.2', messages=messages, tools=ALL_TOOLS)
        messages.append(response.message)
        
        if response.message.tool_calls:
            for tc in response.message.tool_calls:
                func_name = tc.function.name
                args = tc.function.arguments
                print(f"[{step_count}] LLM Decided: {func_name}({args})")
                
                # Execute the actual Python tool
                if func_name in tool_map:
                    try:
                        result = tool_map[func_name](**args)
                        print(f"[{step_count}] Python Observed: {str(result)}")
                        messages.append({'role': 'tool', 'tool_name': func_name, 'content': str(result)})
                    except Exception as e:
                        error_msg = f"Error: Invalid parameters - {str(e)}"
                        print(f"[{step_count}] Python Observed: {error_msg}")
                        messages.append({'role': 'tool', 'tool_name': func_name, 'content': error_msg})
        else:
            print(f"[{step_count}] Final Response: {response.message.content}")
            break

# 1. Test a standard valid request
run_real_agent_test("Search for the movie 'Inception' and get its details.")

# 2. Test unsupported claims (asking for runtime when we don't have it)
run_real_agent_test("Find a Comedy from 1995. What is its runtime and plot?")

# 3. Test invalid parameters (asking for a year as a word, or a massive number)
run_real_agent_test("Filter for Action movies from the year one million.")


=== TRACE FOR: 'Search for the movie 'Inception' and get its details.' ===
[1] LLM Decided: search_movies({'query': 'Inception'})
[1] Python Observed: [{'movieId': 79132, 'title': 'Inception', 'genres': ['Action', 'Crime', 'Drama', 'Mystery', 'Sci-Fi', 'Thriller', 'IMAX'], 'mean': 4.066433566433567, 'count': 143, 'year': 2010.0}]
[2] Final Response: The movie "Inception" is a 2010 science fiction action film directed by Christopher Nolan. It has a mean rating of 4.066433566433567 out of 5, based on 143 reviews, and is classified under the genres of Action, Crime, Drama, Mystery, Sci-Fi, Thriller, and IMAX.

=== TRACE FOR: 'Find a Comedy from 1995. What is its runtime and plot?' ===
[1] LLM Decided: filter_movies({'genre': 'Comedy', 'year': 1995})
[1] Python Observed: [{'movieId': 1, 'title': 'Toy Story', 'genres': ['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy'], 'mean': 3.92093023255814, 'count': 215, 'year': 1995.0}, {'movieId': 3, 'title': 'Grumpier Old Men', 'genres': [

**Observations:**   
The first Trace returns a relatively good answer. The LLM correctly used the `search_movie()` tool to search the database for the requested movie. It correctly returned information on the mean rating, number of ratings, release year and genres, but it also invented additional information not given from the database, namely that it was directed by Christopher Nolan.  

The second Trace revelas a bigger issue. It again correctly used the right tool and observed a large return of possible movies fitting the arguments. It juts chose a random movie out of this set (not for example the one with the highets rating) and invented both the plot and runtime although those are not returned by the tool. This invented information may be correct if it has seen this during training, but this is still a big issue, since the answer should only rely on the information gained from the used tools.  

The third Trace seems to work fine again. The LLM chose the right tool with teh right arguments for which the return is correctly empty. The final response then also states that there possibly is no movie for those arguments. A reason for that may also be the clearly unrealistic release year of the year 1 million


In [7]:
# 4. Test invalid parameters (asking for a year as a word, or a massive number)
run_real_agent_test("Filter movies from the year 1905.")


=== TRACE FOR: 'Filter movies from the year 1905.' ===
[1] LLM Decided: filter_movies({'genre': '', 'year': 1905})
[1] Python Observed: []
[2] Final Response: Based on the tool call response, I was unable to filter any movies from the year 1905. This is because there are no movies released in 1905 that have been digitized or made available for filtering. However, I can suggest some notable films from the early days of cinema that were released around 1905:

* "A Trip to the Moon" (1902) by Georges Méliès
* "The Great Train Robbery" (1903) by Edwin S. Porter
* "The Execution of Mary Stuart" (1909) by Georges Méliès (note: while this film was released in 1909, it is often cited as a precursor to the film "A Trip to the Moon")
* "The Life of an American Fireman" (1903) by Edwin S. Porter

These films are considered to be among the earliest surviving feature-length films, and are still studied and celebrated today for their historical significance and artistic merit.


In this second test, again with a year in which no movie was released that is listed in the database the final response is half correct. It correctly states that there were no movies found in the database, but then the llm again aims to be "overhelpful" and decides to name some movies on its own. Clearly further restrictions (e.g. system prompt, call multiple tools) need to be applied to improve the final responses

#### Improved LLM call with clear restrictions

In [ ]:
def run_real_agent_test(user_prompt):
    # System prompt to enforce strict adherence to tool outputs and prevent hallucination
    system_instruction = """You are a strict, transparent movie recommendation agent. 
    1. You MUST ONLY use information explicitly provided by your tool observations. 
    2. NEVER invent, guess, or hallucinate movie plots, runtimes, actors, or ratings from your own memory. 
    3. If you need details about a specific movie, you MUST use the get_movie_details tool.
    4. If a tool explicitly says a field like runtime or plot is 'Unknown', you must tell the user it is unknown."""

    messages = [
        {'role': 'system', 'content': system_instruction},
        {'role': 'user', 'content': user_prompt}
    ]
    
    tool_map = {f.__name__: f for f in ALL_TOOLS}
    step_count = 0

    print(f"\n=== TRACE FOR: '{user_prompt}' ===")
    
    while step_count < 4:
        step_count += 1
        response = chat(model='llama3.2', messages=messages, tools=ALL_TOOLS)
        messages.append(response.message)
        
        if response.message.tool_calls:
            for tc in response.message.tool_calls:
                func_name = tc.function.name
                args = tc.function.arguments
                print(f"[{step_count}] LLM Decided: {func_name}({args})")
                
                # Execute the actual Python tool
                if func_name in tool_map:
                    try:
                        result = tool_map[func_name](**args)
                        print(f"[{step_count}] Python Observed: {str(result)}") 
                        messages.append({'role': 'tool', 'tool_name': func_name, 'content': str(result)})
                    except Exception as e:
                        error_msg = f"Error: Invalid parameters - {str(e)}"
                        print(f"[{step_count}] Python Observed: {error_msg}")
                        messages.append({'role': 'tool', 'tool_name': func_name, 'content': error_msg})
        else:
            print(f"[{step_count}] Final Response: {response.message.content}")
            break


# 1. Test a standard valid request
run_real_agent_test("Search for the movie 'Inception' and get its details.")

# 2. Test unsupported claims (asking for runtime when we don't have it)
run_real_agent_test("Find a Comedy from 1995. What is its runtime and plot?")

# 3. Test invalid parameters (asking for a year as a word, or a massive number)
run_real_agent_test("Filter movies from the year 1905.")


=== TRACE FOR: 'Search for the movie 'Inception' and get its details.' ===
[1] LLM Decided: search_movies({'query': 'Inception'})
[1] Python Observed: [{'movieId': 79132, 'title': 'Inception', 'genres': ['Action', 'Crime', 'Drama', 'Mystery', 'Sci-Fi', 'Thriller', 'IMAX'], 'mean': 4.066433566433567, 'count': 143, 'year': 2010.0}]
[2] Final Response: Here are the details for the movie 'Inception':

**Movie Title:** Inception
**Movie Genre:** Action, Crime, Drama, Mystery, Sci-Fi, Thriller, IMAX
**Movie Release Year:** 2010
**Movie Rating:** The mean rating given to the movie is 4.066433566433567 out of (a number that is not specified in the output).
**Movie Run Time:** Unknown
**Movie Plot:** Unknown
**Movie Cast:** Unknown

=== TRACE FOR: 'Find a Comedy from 1995. What is its runtime and plot?' ===
[1] LLM Decided: filter_movies({'genre': 'Comedy', 'year': '1995'})
[1] Python Observed: []
[2] Final Response: The search results did not yield a Comedy from 1995. Let me try again using t

**Final Observations and future improvements:**   
The added system prompt somewhat helps. It adds some stability but the model still struggles sometimes. Over all the tool connection works, the llm can call the written tools and execture them. But, the results are unstable (Non-Deterministic). Running the same tests (codecells above) multiple times, results in different returns between the runs. Sometimes the results are perfectly as expected/hoped, sometimes tiny errors by the llm lead to empty, wrong or hallucinated answers. For example in this last test in the codecell above the second trace ("Find a Comedy from 1995. What is its runtime and plot?") shows that the tool call with `filter_movies` returned an empty python response, because the llm put in the argument for year as string instead of integer, although it is specified in the function that the input should be an integer. In the final response it then presented a fake execution of a second tool, with a fake response even formated in a dictionary to try and appease the user. 

A reason for this unstability/randomness and the hallucinations despite the clear sytsem instructions could be the small model used here (llama3.2). According to some research and a chat with Gemini (I hope Gemini's response to my question regarding this issue was not a hallucination...), such smaller models struggle with instruction hierarchy and schema adherence. So juggling conversation, system prompt, tool execution, formatting et. its attention breaks down and starts to hallucinate processes (e.g. a tool call or information) to keep the user "happy" and is, at its core, just predicting the next most logical word. 

Future improvements to fix some of the issues discussed above could be the following:
- Defensive Python Programming: Update the Python tools to be forgiving. For example, explicitly casting `year = int(year)` inside `filter_movies()` would instantly fix the '1995' string bug without relying on the LLM to be perfect.
- Structured Outputs (Pydantic): Instead of relying on basic tool docstrings, use strict JSON schema enforcement paired with Pydantic to physically prevent the LLM from outputting strings when an integer is required.
- Temperature Tuning: Force the agent loop to use temperature=0.0 to eliminate randomness and make tool-calling deterministic and repeatable.
- Upgrade to a bigger model if my notebooks capabilities and observed request runtime allows it
- Feedback Loops (Self-Correction): If a tool fails (like returning an empty list due to a type error), the Python layer should catch the error and explicitly prompt the LLM: "Your tool call failed because you passed a string. Try again with an integer."